pip install sentence-transformers
pip install pinecone 

In [30]:
from pinecone import Pinecone, ServerlessSpec
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import os

In [31]:

env_path = Path(".env").resolve()

load_dotenv(env_path)  # loads vars into os.environ

# Use variables
pinecone_api_key = os.getenv("PINECONE_API_KEY")
print("Loaded key:", bool(pinecone_api_key))

Loaded key: True


In [32]:
# Initialize a Pinecone client with your API key
pc = Pinecone(api_key=pinecone_api_key)

# Create a standard dense index for custom embeddings (dimension=1024)
index_name = "arminplayground"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

In [33]:
records = [
    {
        "id": "chat_001",
        "text": "I had dinner with my mom today. It felt really warm and comforting.",
        "speaker": "user",
        "timestamp": "2026-04-07T19:30:00",
        "date": "2026-04-07",
        "time_of_day": "evening",
        "emotion": {
            "primary": "warm",
            "secondary": ["happy", "safe"],
            "intensity": 0.82
        },
        "tags": ["family", "mom", "daily_life", "positive_moment"],
        "entities": ["mom", "dinner"],
        "related_ids": ["chat_004", "chat_007"],
        "context": {"location": "home", "event": "dinner"},
        "importance": 0.75,
        "embedding": None
    },
    {
        "id": "chat_002",
        "text": "I argued with my friend today and it made me feel really upset.",
        "speaker": "user",
        "timestamp": "2026-04-06T22:10:00",
        "date": "2026-04-06",
        "time_of_day": "night",
        "emotion": {
            "primary": "sad",
            "secondary": ["angry", "frustrated"],
            "intensity": 0.88
        },
        "tags": ["friend", "conflict", "argument", "negative_moment"],
        "entities": ["friend"],
        "related_ids": ["chat_008"],
        "context": {"location": "chat", "event": "argument"},
        "importance": 0.9,
        "embedding": None
    },
    {
        "id": "chat_003",
        "text": "I finally finished my project today. I feel so proud of myself.",
        "speaker": "user",
        "timestamp": "2026-04-05T16:00:00",
        "date": "2026-04-05",
        "time_of_day": "afternoon",
        "emotion": {
            "primary": "proud",
            "secondary": ["happy", "relieved"],
            "intensity": 0.91
        },
        "tags": ["achievement", "work", "success"],
        "entities": ["project"],
        "related_ids": ["chat_009"],
        "context": {"location": "home", "event": "work"},
        "importance": 0.85,
        "embedding": None
    },
    {
        "id": "chat_004",
        "text": "I called my mom today, it was nice hearing her voice.",
        "speaker": "user",
        "timestamp": "2026-04-03T20:15:00",
        "date": "2026-04-03",
        "time_of_day": "evening",
        "emotion": {
            "primary": "happy",
            "secondary": ["warm", "nostalgic"],
            "intensity": 0.7
        },
        "tags": ["family", "mom", "call"],
        "entities": ["mom"],
        "related_ids": ["chat_001"],
        "context": {"location": "phone", "event": "call"},
        "importance": 0.65,
        "embedding": None
    },
    {
        "id": "chat_005",
        "text": "I feel really stressed about my exams next week.",
        "speaker": "user",
        "timestamp": "2026-04-02T23:40:00",
        "date": "2026-04-02",
        "time_of_day": "night",
        "emotion": {
            "primary": "stress",
            "secondary": ["anxious", "worried"],
            "intensity": 0.92
        },
        "tags": ["school", "exam", "stress"],
        "entities": ["exam"],
        "related_ids": [],
        "context": {"location": "home", "event": "study"},
        "importance": 0.95,
        "embedding": None
    },
    {
        "id": "chat_006",
        "text": "Went out with friends today, it was super fun and refreshing.",
        "speaker": "user",
        "timestamp": "2026-04-01T18:20:00",
        "date": "2026-04-01",
        "time_of_day": "evening",
        "emotion": {
            "primary": "happy",
            "secondary": ["excited", "free"],
            "intensity": 0.87
        },
        "tags": ["friends", "social", "fun"],
        "entities": ["friends"],
        "related_ids": [],
        "context": {"location": "outside", "event": "hangout"},
        "importance": 0.7,
        "embedding": None
    },
    {
        "id": "chat_007",
        "text": "Had a quiet dinner alone today, felt a bit lonely.",
        "speaker": "user",
        "timestamp": "2026-03-31T19:00:00",
        "date": "2026-03-31",
        "time_of_day": "evening",
        "emotion": {
            "primary": "lonely",
            "secondary": ["sad"],
            "intensity": 0.78
        },
        "tags": ["alone", "dinner", "emotion"],
        "entities": ["dinner"],
        "related_ids": ["chat_001"],
        "context": {"location": "home", "event": "dinner"},
        "importance": 0.8,
        "embedding": None
    },
    {
        "id": "chat_008",
        "text": "My friend apologized and we made up. I feel relieved now.",
        "speaker": "user",
        "timestamp": "2026-04-07T14:10:00",
        "date": "2026-04-07",
        "time_of_day": "afternoon",
        "emotion": {
            "primary": "relieved",
            "secondary": ["happy"],
            "intensity": 0.83
        },
        "tags": ["friend", "conflict_resolution"],
        "entities": ["friend"],
        "related_ids": ["chat_002"],
        "context": {"location": "chat", "event": "resolution"},
        "importance": 0.88,
        "embedding": None
    }
]


In [34]:
# Real embedding model: BGE large English (1024 dims)
# First run downloads model weights. Cache is moved to D: to avoid low C: space errors.
hf_cache_dir = Path("./.hf-cache").resolve()
hf_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(hf_cache_dir)
os.environ["HF_HUB_CACHE"] = str(hf_cache_dir / "hub")

embed_model = SentenceTransformer(
    "BAAI/bge-large-en-v1.5",
    cache_folder=str(hf_cache_dir),
)

def embed_text(text: str) -> list[float]:
    vec = embed_model.encode(text, normalize_embeddings=True)
    vec = vec.tolist() if hasattr(vec, "tolist") else list(vec)
    if len(vec) != 1024:
        raise ValueError(f"Expected 1024 dims from BGE, got {len(vec)}")
    return vec

for r in records:
    r["embedding"] = embed_text(r["text"])

print("Embeddings generated for", len(records), "records")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\natou\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\natou\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

c:\Users\natou\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:799: UserWarning: Not enough free disk space to download the file. The expected file size is: 1340.62 MB. The target location C:\Users\natou\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5\blobs only has 932.11 MB free disk space.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

OSError: [Errno 28] No space left on device

In [ ]:
# Get index handle
index = pc.Index(index_name)
namespace = "chat-memory"

# Build Pinecone vectors from 1024-d embeddings.
# If a record is missing embedding, generate it from embed_text() automatically.
vectors = []
for r in records:
    emb = r.get("embedding")
    if emb is None:
        emb = embed_text(r.get("text", ""))
        r["embedding"] = emb

    if len(emb) != 1024:
        raise ValueError(f"Record {r['id']} embedding dim is {len(emb)}; expected 1024.")

    vectors.append({
        "id": r["id"],
        "values": emb,
        "metadata": {
            "text": r.get("text", ""),
            "speaker": r.get("speaker"),
            "timestamp": r.get("timestamp"),
            "date": r.get("date"),
            "time_of_day": r.get("time_of_day"),
            "emotion_primary": (r.get("emotion") or {}).get("primary"),
            "emotion_intensity": (r.get("emotion") or {}).get("intensity"),
            "tags": r.get("tags", []),
            "entities": r.get("entities", []),
            "related_ids": r.get("related_ids", []),
            "importance": r.get("importance", 0.0),
        },
    })


In [ ]:
# Upsert all vectors to Pinecone (standard vector API)
index.upsert(vectors=vectors, namespace=namespace)

print(f"Indexed {len(vectors)} vectors into namespace '{namespace}'.")

Indexed 8 vectors into namespace 'chat-memory'.


In [ ]:
# Quick retrieval test (custom embeddings)
query_text = "I felt happy after talking to my mom"
query_embedding = embed_text(query_text)

if len(query_embedding) != 1024:
    raise ValueError(f"query_embedding dim is {len(query_embedding)}; expected 1024.")

results = index.query(
    namespace=namespace,
    vector=query_embedding,
    top_k=3,
    include_metadata=True
)

for i, hit in enumerate(results.get("matches", []), start=1):
    md = hit.get("metadata", {})
    print(f"{i}. id={hit.get('id')} score={hit.get('score'):.4f}")
    print("   text:", md.get("text"))
    print("   emotion:", md.get("emotion_primary"), "importance:", md.get("importance"))
    print()

1. id=chat_005 score=0.0570
   text: I feel really stressed about my exams next week.
   emotion: stress importance: 0.95

2. id=chat_003 score=0.0217
   text: I finally finished my project today. I feel so proud of myself.
   emotion: proud importance: 0.85

3. id=chat_008 score=0.0157
   text: My friend apologized and we made up. I feel relieved now.
   emotion: relieved importance: 0.88

